# Решения: логирование и raise

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import pandas as pd


def _find(name: str) -> Path:
    for p in (Path(name), Path(f'../../data/{name}'), Path(f'../data/{name}')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'{name} не найден — положите slim CSV рядом с ноутбуком')


ORDERS_PATH = _find('orders_slim.csv')
CUSTOMERS_PATH = _find('customers_slim.csv')
PAYMENTS_PATH = _find('payments_slim.csv')

orders = pd.read_csv(ORDERS_PATH, parse_dates=['order_purchase_timestamp'])
if 'order_delivered_customer_date' in orders.columns:
    orders['order_delivered_customer_date'] = pd.to_datetime(
        orders['order_delivered_customer_date'], errors='coerce'
    )
customers = pd.read_csv(CUSTOMERS_PATH)
payments = pd.read_csv(PAYMENTS_PATH)


## Урок. 1-5

In [ ]:
orders_pay = orders.merge(payments, on='order_id', how='left')
log_steps = [
    'loaded orders/customers/payments',
    'merged orders with payments by order_id',
    'parsed purchase timestamp',
    'validated payment_value and mandatory dates',
    'ready for feature engineering',
]

def validate_orders(frame):
    if (frame['payment_value'] < 0).any():
        bad_idx = int(frame.index[frame['payment_value'] < 0][0])
        raise ValueError(f'negative payment_value at row {bad_idx}')
    if frame['order_purchase_timestamp'].isna().any():
        bad_idx = int(frame.index[frame['order_purchase_timestamp'].isna()][0])
        raise ValueError(f'missing order_purchase_timestamp at row {bad_idx}')
    return True


ok = validate_orders(orders_pay)
bad_row = orders_pay.head(3).copy()
bad_row.loc[bad_row.index[0], 'payment_value'] = -10
try:
    validate_orders(bad_row)
except ValueError as e:
    error_text = str(e)
bad_dates = orders_pay.head(5).copy()
bad_dates.loc[bad_dates.index[0], 'order_purchase_timestamp'] = pd.NaT
try:
    validate_orders(bad_dates)
except ValueError as e:
    error_date = str(e)
LOG_NOTE = (
    'Лог делает pipeline воспроизводимым: видно порядок шагов и место сбоя. '
    'Raise останавливает обработку на невалидных данных до того, как ошибка попадёт в признаки и отчёт.'
)
print(ok)
print(log_steps)
print(error_text)
print(error_date)
print(LOG_NOTE)

## ДЗ. 1-3

In [ ]:
def validate_payments(frame):
    if frame['payment_value'].isna().any():
        raise ValueError('payment_value has missing values')
    if (frame['payment_value'] < 0).any():
        raise ValueError('payment_value must be non-negative')
    return True


result = validate_payments(payments)
bad = payments.head(3).copy()
bad.loc[bad.index[0], 'payment_value'] = -1
try:
    validate_payments(bad)
except ValueError as e:
    msg = str(e)
CONTRACT_NOTE = (
    'Контракт фиксирует, какие данные pipeline считает допустимыми. '
    'Без этого модель может учиться на испорченных строках, и ошибка проявится позже, '
    'когда уже непонятно, на каком шаге она возникла.'
)
print(result)
print(msg)
print(CONTRACT_NOTE)